# Lab 15b: Microsoft Agent Framework with Tracing (Direct APIM)

This notebook demonstrates how to trace agents built with **Microsoft Agent Framework (MAF)** using direct APIM calls.

## Prerequisites

Run `main.ipynb` first to deploy the required resources.

## What You'll Learn

| Concept | Description |
|---------|-------------|
| **setup_observability** | Configure tracing with App Insights connection string |
| **AzureOpenAIChatClient** | MAF client for Azure OpenAI API format |
| **ChatAgent** | Create agents using MAF's `ChatAgent` class |
| **Direct APIM** | Call APIM gateway directly without Foundry hosting |

## References

- [MAF Observability Guide](https://learn.microsoft.com/azure/ai-foundry/observability/how-to/trace-agent-framework?view=foundry)
- [OpenTelemetry GenAI Semantic Conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/)

In [ ]:
# Install dependencies
%pip install -q agent-framework azure-identity openai
%pip install -q azure-monitor-opentelemetry
%pip install -q opentelemetry-api opentelemetry-sdk

In [ ]:
import json
from pathlib import Path

# Load configuration from infrastructure deployment
config_file = Path("spoke-config.json")
if not config_file.exists():
    raise FileNotFoundError("Run main.ipynb first to deploy resources")

config = json.loads(config_file.read_text())

APP_INSIGHTS_CONN_STRING = config["APP_INSIGHTS_CONN_STRING"]
APP_INSIGHTS_NAME = config["APP_INSIGHTS_NAME"]
RESOURCE_GROUP = config["RESOURCE_GROUP"]
APIM_URL = config["APIM_URL"]
APIM_KEY = config["APIM_KEY"]
MODEL_NAME = config["GATEWAY_MODEL"]

# Agent configuration
AGENT_NAME = "NASASpaceFactsAgent-MAF"
AGENT_ID = "nasa-space-facts-maf"  # OpenTelemetry agent ID

print("Configuration loaded")
print(f"  App Insights: {APP_INSIGHTS_NAME}")
print(f"  APIM URL: {APIM_URL}")
print(f"  Model: {MODEL_NAME}")

---
## Step 1: Configure Observability

MAF provides `setup_observability()` which automatically configures Azure Monitor when you pass the `applicationinsights_connection_string` parameter. This enables all MAF spans.

In [33]:
from agent_framework.observability import setup_observability

# Configure observability with App Insights - this sets up Azure Monitor automatically
setup_observability(
    applicationinsights_connection_string=APP_INSIGHTS_CONN_STRING,
    enable_sensitive_data=True,  # Record prompts/responses in traces
)

print("Observability configured with Azure Monitor")
print("  Sensitive data recording: enabled")

Observability configured with Azure Monitor
  Sensitive data recording: enabled


---
## Step 2: Create Chat Client for APIM

APIM uses Azure OpenAI API format (`/deployments/{model}/chat/completions`), so we use `AzureOpenAIChatClient`.

In [ ]:
from agent_framework.azure import AzureOpenAIChatClient


chat_client = AzureOpenAIChatClient(
    api_key=APIM_KEY,
    endpoint=APIM_URL.replace("/openai", ""),
    deployment_name=MODEL_NAME,
    api_version="2024-10-21",
)

print(f"Chat client configured for APIM")
print(f"  Endpoint: {APIM_URL.replace('/openai', '')}")
print(f"  Model: {MODEL_NAME}")

---
## Step 3: Define the Space Facts Tool

MAF uses Python functions as tools. We'll create a `get_space_fact` function.

In [35]:
# Space facts database
SPACE_FACTS = {
    "Mars": "Mars has the largest volcano in the solar system - Olympus Mons, standing 72,000 feet tall (nearly 3x Mount Everest). A day on Mars is 24 hours 37 minutes!",
    "James Webb": "The James Webb Space Telescope's mirror is 6.5 meters wide and made of gold-plated beryllium. It operates at -233C and can see galaxies formed 13.5 billion years ago!",
    "black holes": "The largest known black hole, TON 618, has a mass of 66 billion suns. Its event horizon is larger than our entire solar system!",
    "Saturn": "Saturn's density is so low that if you could find a bathtub big enough, it would float! Its rings span 282,000 km but are only about 10 meters thick.",
    "ISS": "The International Space Station travels at 17,500 mph, orbiting Earth every 90 minutes. Astronauts see 16 sunrises and sunsets every day!",
}

def get_space_fact(topic: str) -> str:
    """Get detailed information about a space topic, planet, or NASA mission.
    
    Args:
        topic: The space topic to get facts about (e.g., 'Mars', 'James Webb', 'black holes')
    
    Returns:
        A fascinating fact about the requested topic
    """
    for key, fact in SPACE_FACTS.items():
        if key.lower() in topic.lower() or topic.lower() in key.lower():
            return fact
    
    return f"Space is vast and mysterious! The observable universe contains about 2 trillion galaxies. {topic} is an exciting topic - there's always more to discover!"

print("Space facts tool defined")
print(f"  Available topics: {list(SPACE_FACTS.keys())}")

Space facts tool defined
  Available topics: ['Mars', 'James Webb', 'black holes', 'Saturn', 'ISS']


---
## Step 4: Create the NASA Space Facts Agent

Using MAF's `ChatAgent` with the OpenTelemetry agent ID for trace correlation.

In [36]:
from agent_framework import ChatAgent

# Agent instructions
SPACE_AGENT_INSTRUCTIONS = """You are a NASA Space Facts expert assistant.
Your mission is to share fascinating facts about space exploration, planets, stars, 
galaxies, and NASA missions. When users ask about specific celestial bodies or missions,
use the get_space_fact function to provide detailed information.

Be enthusiastic and educational. Include fun comparisons to help users understand scale."""

# Create the agent with OpenTelemetry ID for trace correlation
space_agent = ChatAgent(
    chat_client=chat_client,
    name=AGENT_NAME,
    instructions=SPACE_AGENT_INSTRUCTIONS,
    tools=[get_space_fact],
    id=AGENT_ID,  # OpenTelemetry agent ID for trace correlation
)

print(f"Created agent: {space_agent.name}")
print(f"  OpenTelemetry ID: {space_agent.id}")
print(f"  Model: {MODEL_NAME}")
print(f"  Tools: get_space_fact")

Created agent: NASASpaceFactsAgent-MAF
  OpenTelemetry ID: nasa-space-facts-maf
  Model: gpt-4.1-mini
  Tools: get_space_fact


---
## Step 5: Invoke the Agent
MAF automatically creates spans for:
- `invoke_agent <agent_name>` - Top level span
- `chat <model_name>` - LLM calls
- `execute_tool <function_name>` - Tool executions

In [37]:
import asyncio
import time

# Queries to test
queries = [
    "Tell me a fascinating fact about Mars!",
    "What is the James Webb Space Telescope discovering?",
    "How big is the largest known black hole?",
]

async def run_agent_queries():
    print("Invoking NASA Space Facts agent (traces captured automatically)...\n")
    
    for query in queries:
        print(f"User: {query}")
        
        start = time.time()
        response = await space_agent.run(query)  # run() is the async method
        duration = (time.time() - start) * 1000
        
        print(f"Agent: {response}")
        print(f"  Duration: {duration:.0f}ms\n")
        
        await asyncio.sleep(1)  # Small delay between requests
    
    print("Agent invocations complete!")

# Run the async function
await run_agent_queries()

Invoking NASA Space Facts agent (traces captured automatically)...

User: Tell me a fascinating fact about Mars!
Agent: Here's a fascinating fact about Mars: It is home to the largest volcano in the entire solar system, Olympus Mons, which stands an astounding 72,000 feet tall—nearly three times the height of Mount Everest! Also, a day on Mars is just a bit longer than Earth's, lasting 24 hours and 37 minutes. Imagine spending a day on a planet with such a massive volcano and slightly longer days! Would you like to hear about other amazing features of Mars or something else?
  Duration: 2899ms

User: What is the James Webb Space Telescope discovering?
Agent: The James Webb Space Telescope has an impressive 6.5-meter-wide mirror made of gold-plated beryllium, designed to operate at a chilly -233°C to capture incredibly detailed images. It's so powerful that it can observe galaxies that formed 13.5 billion years ago, giving us a glimpse into the very early universe! Just imagine peering 

---
## Step 6: Flush Traces

Force flush to ensure all traces are sent to Application Insights.

In [38]:
from opentelemetry import trace as otel_trace
from opentelemetry.sdk.trace import TracerProvider

# Force flush all pending traces
print("Flushing traces to Application Insights...")
provider = otel_trace.get_tracer_provider()
if isinstance(provider, TracerProvider):
    provider.force_flush()
print("Traces flushed")

# Wait for ingestion
print("\nWaiting 30 seconds for trace ingestion...")
time.sleep(30)

Flushing traces to Application Insights...
Traces flushed

Waiting 30 seconds for trace ingestion...


---
## Step 7: View Your Traces

MAF traces appear in the **Grafana Agent Framework Dashboard** in Application Insights.

In [ ]:
import subprocess

# Get subscription info for portal links
sub_result = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True)
SUBSCRIPTION_ID = sub_result.stdout.strip()

tenant_result = subprocess.run('az account show --query tenantId -o tsv', shell=True, capture_output=True, text=True)
TENANT_ID = tenant_result.stdout.strip()

# Grafana MAF Dashboard URL (Agent Framework specific)
GRAFANA_MAF_URL = (
    f"https://ms.portal.azure.com/#view/Microsoft_Azure_Monitoring/AzureGrafana.ReactView"
    f"/GalleryType/microsoft.insights%2Fcomponents"
    f"/ResourceId/%2Fsubscriptions%2F{SUBSCRIPTION_ID}%2FresourceGroups%2F{RESOURCE_GROUP}"
    f"%2Fproviders%2FMicrosoft.Insights%2Fcomponents%2F{APP_INSIGHTS_NAME}"
    f"/ConfigurationId/AgentFramework"
)

print("="*70)
print("WHERE TO VIEW YOUR MAF TRACES")
print("="*70)

print("\n1. GRAFANA - Agent Framework Dashboard")
print("   Purpose-built dashboard for MAF agents with metrics and traces:")
print(f"   {GRAFANA_MAF_URL}")

---
## Summary

| Step | What Happens |
|------|---------------|
| 1. Configure Observability | `setup_observability(applicationinsights_connection_string=...)` |
| 2. Create Chat Client | `AzureOpenAIChatClient` pointing to APIM gateway |
| 3. Define Tools | Python functions with type hints |
| 4. Create Agent | `ChatAgent` with OpenTelemetry ID |
| 5. Run Agent | `await agent.run(query)` - traced automatically |
| 6. Flush | Force send traces to App Insights |
| 7. View | Grafana Agent Framework Dashboard |